# Same-hardware timing — BraTS 2020 activation-function ablation

Measures **per-epoch training time**, **inference time per volume** and **peak VRAM** for **all 12 activation functions on one GPU**, so that the computational-cost comparison needs no cross-device normalisation.

This is timing-only, not full training. Each function runs a few epochs; the **first epoch is discarded** (one-off data-caching / CUDA warm-up cost) and the reported time is the **median of the remaining epochs**. Total runtime is roughly 40 minutes on an A100.

| Setting | Value |
|---|---|
| Functions | all 12 |
| Epochs per function | 5 (epoch 1 discarded → median of 4) |
| Pipeline | identical (model, loss, optimiser, batch size, AMP) |
| Output | `timing_summary.csv` — per-function s/epoch, ms/volume, peak VRAM, on one named GPU |

The device name is recorded in the output so the comparison is transparent. No checkpoints are saved.

## 1 · Mount and configure

In [ ]:
from google.colab import drive
import os, torch
drive.mount('/content/drive', force_remount=True)
DATA_DIR = "/content/drive/MyDrive/BraTS2020_Preprocessed_128"
RESULTS_ROOT = "/content/drive/MyDrive/BraTS_Timing_Results"; os.makedirs(RESULTS_ROOT, exist_ok=True)
ACTIVATIONS = ["relu","leaky_relu","prelu","elu","gelu","swish","mish","elish","hard_elish","logish","smish","tanhexp"]
N_EPOCHS = 5          # epoch 1 discarded -> median of the remaining 4
N_INFER  = 11         # first inference discarded -> 10 timed volumes
BATCH_SIZE = 2; LR = 3e-4
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("GPU for this timing run:", GPU)
print(f"{len(ACTIVATIONS)} functions x {N_EPOCHS} epochs (~40 min total)")

## 2 · Components (identical pipeline)

In [ ]:
import time, random, glob, numpy as np, pandas as pd
import torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

class Mish(nn.Module):
    def forward(self,x): return x*torch.tanh(F.softplus(x))
class ELiSH(nn.Module):
    def forward(self,x): return F.elu(x)*torch.sigmoid(x)
class HardELiSH(nn.Module):
    def forward(self,x): return F.elu(x)*F.hardsigmoid(x)
class Logish(nn.Module):
    def forward(self,x): return x*torch.log(1+torch.sigmoid(x))
class Smish(nn.Module):
    def forward(self,x): return x*torch.tanh(torch.log(1+torch.sigmoid(x)))
class TanhExp(nn.Module):
    def forward(self,x): return x*torch.tanh(torch.exp(torch.clamp(x,max=20)))
def get_activation(n):
    n=n.lower()
    return {'relu':nn.ReLU(inplace=True),'leaky_relu':nn.LeakyReLU(0.01,inplace=True),'prelu':nn.PReLU(),
            'elu':nn.ELU(inplace=True),'gelu':nn.GELU(),'swish':nn.SiLU(inplace=True),'mish':Mish(),
            'elish':ELiSH(),'hard_elish':HardELiSH(),'logish':Logish(),'smish':Smish(),'tanhexp':TanhExp()}[n]

class ResidualBlock(nn.Module):
    def __init__(s,i,o,a):
        super().__init__(); s.c1=nn.Conv3d(i,o,3,padding=1); s.b1=nn.BatchNorm3d(o); s.a=get_activation(a)
        s.c2=nn.Conv3d(o,o,3,padding=1); s.b2=nn.BatchNorm3d(o); s.sk=nn.Conv3d(i,o,1) if i!=o else nn.Identity()
    def forward(s,x): return s.a(s.b2(s.c2(s.a(s.b1(s.c1(x)))))+s.sk(x))
class ImprovedUNet3D(nn.Module):
    def __init__(s,ic,oc,a):
        super().__init__(); s.e1=ResidualBlock(ic,32,a); s.p=nn.MaxPool3d(2)
        s.e2=ResidualBlock(32,64,a); s.e3=ResidualBlock(64,128,a); s.e4=ResidualBlock(128,256,a); s.bn=ResidualBlock(256,512,a)
        s.u4=nn.ConvTranspose3d(512,256,2,2); s.d4=ResidualBlock(512,256,a)
        s.u3=nn.ConvTranspose3d(256,128,2,2); s.d3=ResidualBlock(256,128,a)
        s.u2=nn.ConvTranspose3d(128,64,2,2);  s.d2=ResidualBlock(128,64,a)
        s.u1=nn.ConvTranspose3d(64,32,2,2);   s.d1=ResidualBlock(64,32,a); s.o=nn.Conv3d(32,oc,1)
    def forward(s,x):
        e1=s.e1(x); e2=s.e2(s.p(e1)); e3=s.e3(s.p(e2)); e4=s.e4(s.p(e3)); b=s.bn(s.p(e4))
        d4=s.d4(torch.cat([s.u4(b),e4],1)); d3=s.d3(torch.cat([s.u3(d4),e3],1))
        d2=s.d2(torch.cat([s.u2(d3),e2],1)); d1=s.d1(torch.cat([s.u1(d2),e1],1)); return s.o(d1)

class DS(Dataset):
    def __init__(s,d): s.f=sorted(glob.glob(f"{d}/*.npy"))
    def __len__(s): return len(s.f)
    def __getitem__(s,i):
        img=np.load(s.f[i]).astype(np.float32); m=np.load(s.f[i].replace('/images/','/masks/')).astype(np.longlong)
        return torch.from_numpy(img).permute(3,2,0,1), torch.from_numpy(m).permute(2,0,1)
device='cuda' if torch.cuda.is_available() else 'cpu'
print("ready on", device)

## 3 · Time all 12 functions

In [ ]:
train_dl=DataLoader(DS(f"{DATA_DIR}/train/images"),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
val_dl  =DataLoader(DS(f"{DATA_DIR}/val/images"),batch_size=1,shuffle=False,num_workers=2)

def sync():
    if device=='cuda': torch.cuda.synchronize()

def time_one(act):
    seed_everything(42)
    model=ImprovedUNet3D(4,4,act).to(device); opt=optim.AdamW(model.parameters(),lr=LR)
    scaler=torch.amp.GradScaler('cuda'); ce=nn.CrossEntropyLoss()
    torch.cuda.reset_peak_memory_stats()
    ep_times=[]
    model.train()
    for ep in range(N_EPOCHS):
        sync(); t0=time.time()
        for x,y in train_dl:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                p=model(x); ps=F.softmax(p,1); yo=F.one_hot(y,4).permute(0,4,1,2,3).float()
                inter=(ps*yo).sum((2,3,4)); union=ps.sum((2,3,4))+yo.sum((2,3,4))
                loss=ce(p,y)+(1-((2*inter+1e-5)/(union+1e-5)).mean())
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sync(); ep_times.append(time.time()-t0)
    train_med=float(np.median(ep_times[1:]))     # discard epoch 1 (warm-up)
    vram=torch.cuda.max_memory_allocated()/1e9
    # inference timing per volume
    model.eval(); inf=[]
    with torch.no_grad():
        for i,(x,_) in enumerate(val_dl):
            if i>=N_INFER: break
            x=x.to(device); sync(); t0=time.time()
            with torch.amp.autocast('cuda'): _=model(x)
            sync(); inf.append((time.time()-t0)*1000)
    inf_ms=float(np.mean(inf[1:]))               # discard first
    return train_med, inf_ms, vram, ep_times

rows=[]
for act in ACTIVATIONS:
    print(f"timing {act} ...", end=" ", flush=True)
    tm, ims, vram, eps = time_one(act)
    print(f"{tm:.1f}s/epoch | {ims:.1f} ms/vol | {vram:.2f} GB (epochs: {[round(e) for e in eps]})")
    rows.append({'Activation':act,'GPU':GPU,'sec_per_epoch_median':round(tm,2),
                 'inference_ms_per_volume':round(ims,2),'peak_vram_gb':round(vram,2)})
    del eps
df=pd.DataFrame(rows)
df.to_csv(f"{RESULTS_ROOT}/timing_summary.csv",index=False)

print("\n"+"="*70); print(f"  SAME-HARDWARE TIMING — all on: {GPU}"); print("="*70)
print(f"{'Activation':<12} | {'s/epoch':>9} | {'rel. to ReLU':>12} | {'ms/volume':>10} | {'VRAM GB':>8}")
base=df[df.Activation=='relu']['sec_per_epoch_median'].iloc[0]
for _,r in df.iterrows():
    print(f"{r['Activation']:<12} | {r['sec_per_epoch_median']:>9.1f} | {r['sec_per_epoch_median']/base:>11.2f}x | {r['inference_ms_per_volume']:>10.1f} | {r['peak_vram_gb']:>8.2f}")
print(f"\nSaved: {RESULTS_ROOT}/timing_summary.csv")

## 4 · Output
`timing_summary.csv` from `BraTS_Timing_Results/` is archived in this repository under `results/timing/` (Table 9, Figure 13 and Supplementary Table S2 of the paper).